# RSNA Knee Abnormality Detection: Sistem Mimarisi ve Yöntemler Özeti

---

## 1. Genel Bakış ve Yönetici Özeti (Executive Summary)

RSNA Knee Abnormality Detection yarışması, manyetik rezonans (MR) görüntüleri üzerinden diz eklemine ait **12 farklı anatomik ve patolojik bulguyu** (bağ yırtıkları, menisküs lezyonları, kıkırdak aşınması, eklem sıvısı ve kemik kontüzyonları) inceleme (*study*) düzeyinde tespit etmeyi amaçlayan çok etiketli (*multi-label*) bir tıbbi yapay zekâ problemidir.

Projede karşılaşılan iki temel darboğaz ve geliştirilen çözümler:

1. **Etiket Kısıtı (The 58-Label Bottleneck):**  
   Yarışmada yer alan 4.407 hasta incelemesinden yalnızca 58 adedinde uzman radyologlar tarafından işaretlenmiş zemin gerçeği (*ground-truth*) etiketler bulunmaktadır. Kalan 4.349 hastanın yalnızca serbest metinli çok dilli (İngilizce, Fransızca, İspanyolca, Almanca) radyoloji raporları mevcuttur. Test kümesinde ise hiçbir rapor bulunmayıp yalnızca MR görüntüleri yer almaktadır.  
   * **Çözüm:** Çok dilli büyük dil modelleri (LLM) ile rapor metinleri taranmış, 58 altın vaka üzerinde 0.8927 Macro AUC ile doğrulanan en güçlü damıtma modeli seçilerek 4.407 hastanın tamamı için güvenilir zayıf denetimli etiket matrisi (`data/train_annotated.csv`) üretilmiştir.

2. **Görüntüleme ve Hacim Karmaşıklığı (Volumetric Imaging Challenge):**  
   Her hasta incelemesi 3 farklı anatomik düzlemde (Sagital, Koronal, Aksiyel) çekilmiş onlarca kesitten ve farklı tarayıcı matris boyutlarından ($960 \times 960$, $800 \times 800$ vb.) oluşmaktadır. Bir bağ yırtığı 60 kesit içinde yalnızca 1–2 kesitte izlenebilmektedir.  
   * **Çözüm:** Görüntüler çift doğrusal enterpolasyonla $384 \times 384$ ölçeğine normalize edilmiş; ardışık kesitler 2.5D RGB temsillerine dönüştürülmüştür. 12 patolojinin her biri için bağımsız dikkat ağırlıkları hesaplayan **12 Başlıklı Bulguya Özel Dikkat Havuzlama (*Finding-Specific Attention Pooling*)** mimarisi kurularak ortalama havuzlamanın yarattığı sinyal seyreltme etkisi ortadan kaldırılmıştır.

---

## 2. Yöntemler ve Literatür Referansları

| Boru Hattı Aşaması | Kullanılan Yöntem | Teknik ve Matematiksel Rolü | Literatür / Kaynak |
| :--- | :--- | :--- | :--- |
| **Zayıf Denetim** | LLM Tabanlı Rapor Damıtma (*Weak Supervision*) | Çok dilli serbest metinli radyoloji raporlarından 12 yapılandırılmış etiket çıkararak 58 altın vakayı 4.407 vakaya genişletme. | *Rajpurkar et al. (2018), "CheXpert: A Large Chest Radiograph Dataset with Uncertainty Labels and Expert Comparison", AAAI.* |
| **Ön İşleme** | Donanımsal DICOM Kalibrasyonu | `RescaleSlope` ve `RescaleIntercept` değerleri ile fiziksel MR sinyalini kalibre etme; `MONOCHROME1` inversiyonu. | *DICOM PS 3.3 (2023), "Digital Imaging and Communications in Medicine: Information Object Definitions", NEMA.* |
| **Ön İşleme** | Persentil Normalizasyonu (%1-%99 Clipping) | Ekstrem yağ dokusu ve metalik artefaktları ayıklayıp doku kontrastını koruyarak $[0.0, 1.0]$ aralığına sıkıştırma. | *Nyúl & Udupa (1999), "On standardizing the MR image intensity scale", IEEE TMI.* |
| **Ön İşleme** | Periferik Dilim Budaması (%10-%90 Air Trimming) | Eklem dışındaki boş hava ve cilt kesitlerini eleyerek GPU hesaplama bütçesini eklem yüzeyine odaklama. | *Bien et al. (2018), "Deep-learning-assisted diagnosis for knee magnetic resonance imaging (MRNet)", PLOS Medicine.* |
| **Uzamsal Standartlaştırma** | Çift Doğrusal Enterpolasyon (*Bilinear Interpolation*) | Farklı hastane cihazlarından gelen değişken çözünürlükleri ($960 \times 960$, $800 \times 800$) homojen $(384, 384)$ matrisine indirgeme. | *Gonzalez & Woods (2018), "Digital Image Processing (4th Edition)", Pearson.* |
| **Temsil Öğrenme** | 2.5D Çok Düzlemli Dilim Yığınlama (*Slice Stacking*) | $[Slice_{i-1}, Slice_i, Slice_{i+1}]$ üçlüsünü RGB kanalları gibi birleştirerek 3D bağlamı 2D omurga hızıyla koruma. | *Roth et al. (2014), "A new 2.5D representation for lymph node detection using random sets of deep convolutional networks", MICCAI.* |
| **Öznitelik Çıkarımı** | Artık Evrişimli Omurga (*ResNet-34*) | ImageNet üzerinde önceden eğitilmiş 21M parametreli derin artık ağ ile pencerelerden 512 boyutlu görsel vektör çıkarma. | *He et al. (2016), "Deep Residual Learning for Image Recognition", CVPR.* |
| **Görünüm Entegrasyonu** | Düzlem Koşullama (*Plane Embedding*) | Sagital (0), Koronal (1) ve Aksiyel (2) açılarını $d=512$ boyutlu öğrenilebilir vektörlerle görsel özniteliklere ekleme. | *Vaswani et al. (2017), "Attention Is All You Need", NeurIPS.* |
| **Havuzlama & Karar** | 12 Başlı Dikkat Havuzlaması (*Finding-Specific Attention*) | Her patoloji için $a_{k, i} = \mathbf{w}_k^T \tanh(V f_i)$ formülüyle bağımsız dikkat ağırlıkları hesaplayıp ilgili kesiti büyütme. | *Ilse, Tomczak, & Welling (2018), "Attention-based Deep Multiple Instance Learning", ICML.* |
| **Kayıp Fonksiyonu** | Pozitif Ağırlıklı İkili Çapraz Entropi (*Finding-Weighted BCE*) | Nadir sınıfları (MCL %15) sık sınıflara (Efüzyon %60) karşı $w_k = \min(4.0, \max(1.0, N_{\text{neg}}/N_{\text{pos}}))$ ile dengeleme. | *Lin et al. (2017), "Focal Loss for Dense Object Detection", ICCV / IEEE TPAMI.* |
| **Optimizasyon** | Ayrıştırılmış Ağırlık Çürümesi (*AdamW*) | L2 regülarizasyonunu gradyan momentlerinden ayırarak derin katmanlarda genelleme yeteneğini maksimize etme. | *Loshchilov & Hutter (2017), "Decoupled Weight Decay Regularization (AdamW)", ICLR.* |
| **Öğrenme Takvimi** | Kosinüs Sönümleme (*Cosine Annealing LR*) | Öğrenme oranını $10^{-4}$'ten $10^{-6}$'ya sönümleyerek modelin kararlı yerel minimumlara pürüzsüz oturmasını sağlama. | *Loshchilov & Hutter (2016), "SGDR: Stochastic Gradient Descent with Warm Restarts", ICLR.* |
| **Donanım Hızlandırma** | Otomatik Karma Duyarlık (*AMP FP16*) | 16-bit kayan nokta tensör çekirdekleri ile GPU bellek kullanımını yarıya indirip hesaplama hızını 2 katına çıkarma. | *Micikevicius et al. (2018), "Mixed Precision Training", ICLR.* |
| **Doğrulama** | Tabakalı 5 Katlı Çapraz Doğrulama (*Stratified K-Fold*) | 58 altın vaka ve 4.349 zayıf vakayı hasta düzeyinde (`StudyInstanceUID`) sızıntısız 5 eşit katlamaya paylaştırma. | *Sechidis, Tsoumakas, & Vlahavas (2011), "On the stratification of multi-label data", ECML PKDD.* |
| **Değerlendirme** | Çok Etiketli Macro ROC-AUC | 12 patolojinin ayrı ayrı ROC eğrisi altındaki alanlarını hesaplayıp aritmetik ortalamasını alarak resmi skoru üretme. | *Fawcett (2006), "An introduction to ROC analysis", Pattern Recognition Letters.* |

---

## 3. Sistem Mimarisi Akış Şeması

```
[Ham DICOM Verisi: 22.035 Seri]
          │
          ▼  (Fiziksel Kalibrasyon + %1-%99 Persentil Kırpma + Hava Budama)
[Standartlaştırılmış 2D Dilimler (384 × 384)]
          │
          ▼  (2.5D Dilim Yığınlama: Sagital, Koronal, Aksiyel)
[Pencere Matrisi (N, 3, 384, 384)]
          │
          ▼  (ResNet-34 Feature Extractor + Plane Embedding)
[Görsel Öznitelik Tensörleri (N, 512)]
          │
          ▼  (12-Head Finding-Specific Attention Pooling)
[12 Ayrı Klinik Bağlam Vektörü (c_k)]
          │
          ▼  (Finding-Weighted BCE + AdamW + Cosine Annealing)
[Çıktı Logitleri ve Tahmin Olasılıkları (1, 12)]
          │
          ▼
[Doğrulanmış submission.csv (Macro ROC-AUC Hedefi: 0.85 - 0.92+)]
```

> **Özet:** Bu boru hattı, literatürdeki Çoklu Örnek Öğrenimi (*Multiple Instance Learning - MIL*) ilkeleri ile tıbbi görüntüleme ön işleme standartlarını bir araya getiren; bellek sızıntısı olmayan, donanım düzeyinde optimize edilmiş ve yarışma şartlarına hazır bir uçtan uca mimaridir.

#### İçindekiler Tablosu 
1. **Bölüm 1: Ortam Kurulumu, Kütüphaneler ve Hiperparametre Konfigürasyonu**
    1. Kütüphane İçe Aktarımları ve Donanım Denetimi (CUDA / GPU)
    2. Hiperparametre Sınıfı (`Config`) ve Otomatik Dizin Çözümleme
2. **Bölüm 2: Dinamik Veri Girişi ve 4.407 Vakalık Kohort Yönetimi**
    1. Çok Kaynaklı Dizin Taraması (`load_training_cohort`)
    2. Zayıf Denetimli Pseudo-Label Entegrasyonu ve Altın Standart İzolasyonu
3. **Bölüm 3: Fiziksel DICOM İşleme Motoru ve 2.5D Pencereleme**
    1. Eğim/Kesişim (Slope/Intercept) Kalibrasyonu ve Percentile Normalizasyonu
    2. Değişken MR Çözünürlüklerinin Çift Çizgisel (Bilinear) Standartlaştırılması
    3. Üç Düzlemli (Sagittal, Coronal, Axial) 2.5D RGB Dilim Yığınlama
4. **Bölüm 4: Veri Kümesi (Dataset) ve Tabakalı Katlama (Stratification)**
    1. Hasta Düzeyinde `KneeStudyDataset` Boru Hattı
    2. Değişken Dilim Sayılarını Yöneten Özel Birleştirici (`collate_studies`)
    3. 5 Katlı Tabakalı Bölümleme (Stratified 5-Fold Partitioning)
5. **Bölüm 5: Çok Görünümlü Derin Öğrenme Mimarisi: 12 Başlı Dikkat Havuzlama**
    1. Evrişimli Omurga (ResNet-34) ve Düzlem Koşullama (Plane Embedding)
    2. Patolojiye Özel 12 Başlı Dikkat Havuzlama (`FindingSpecificAttentionPooling`)
    3. Matematiksel İleri Besleme Akışı
6. **Bölüm 6: Kayıp Fonksiyonu, Optimizasyon ve Eğitim Döngüsü**
    1. Sınıf Dengesizliklerine Duyarlı Finding-Weighted BCE Kayıp Fonksiyonu
    2. AdamW Optimizer ve Kosinüs Sönümlemeli Öğrenme Oranı Takvimi
    3. Karma Duyarlık (AMP FP16) ve Gradyan Biriktirme ile Epoch Eğitimi
7. **Bölüm 7: Doğrulama Kulesi, Macro ROC-AUC ve Model Kaydı**
    1. Çok Etiketli Macro ROC-AUC Hesaplama Motoru
    2. En İyi Modelin (`best_model.pt`) Tespiti ve Kaggle Çıktı Kaydı

## Bölüm 1: Ortam Kurulumu, Kütüphaneler ve Hiperparametre Konfigürasyonu
Bu bölüm, çalışma ortamının medikal veri işleme ve derin öğrenme gereksinimlerini hazırlar, kullanılabilir GPU kaynaklarını doğrular ve boru hattının tüm dinamik yollarını belirler.

### 1.1 Hücre: Kütüphanelerin Yüklenmesi ve Donanım Denetimi

In [ ]:
import gc
import os
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
import torchvision.models as tv_models

try:
    import pydicom
except ImportError:
    pydicom = None

try:
    import timm
except ImportError:
    timm = None

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")

> *Mühendislik Açıklaması* <br>
    1. **PyDICOM ve Timm Kontrolü:** Kütüphaneler ortamda bulunmadığında sistemin çökmesini engellemek için koruyucu `try-except` blokları kullanılmıştır.<br>
    2. **CUDA Denetimi:** `torch.cuda.is_available()` çağrısı, hesaplamaların Kaggle üzerindeki NVIDIA T4 veya P100 hızlandırıcısına taşınabileceğini garanti eder.

### 1.2 Hücre: Hiperparametre Sınıfı (`Config`) ve Dinamik Dizin Çözümleme
Hedef 12 anormallik etiketinin sırası, görüntü çözünürlüğü, parti boyutu (batch size), 
öğrenme oranı ve dinamik veri yolu tespiti bu sınıfta merkezi olarak yönetilir

In [ ]:
class Config:
    TARGET_COLS = [
        'ACL',
        'MCL',
        'Medial Meniscus',
        'Lateral Meniscus',
        'Medial OA',
        'Lateral OA',
        'PF OA',
        'Effusion',
        'Synovitis',
        "Baker's",
        'Contusion',
        'Fracture'
    ]
    NUM_CLASSES = len(TARGET_COLS)  # 12
    
    # Training hyperparameters
    BACKBONE = 'resnet34'
    PRETRAINED = True
    IMAGE_SIZE = 384
    MAX_WINDOWS_PER_STUDY = 48
    BATCH_SIZE = 2
    GRAD_ACCUM_STEPS = 2  # Effective batch size: 4
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-2
    EPOCHS = 5
    NUM_FOLDS = 5
    ACTIVE_FOLD = 0
    NUM_WORKERS = 2
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('../experiments')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_competition_path() -> Path:
    """Finds the directory containing train.csv and train_series.csv."""
    if Path('/kaggle/input').exists():
        for p in Path('/kaggle/input').rglob('train.csv'):
            if (p.parent / 'train_series.csv').exists():
                return p.parent
        p_std = Path('/kaggle/input/rsna-knee-abnormality-detection')
        if p_std.exists():
            return p_std
    for loc in [Path('data'), Path('../data'), Path('../../data'), Path('.')]:
        if (loc / 'train.csv').exists():
            return loc.resolve()
    return Path('data').resolve()

Config.DATA_DIR = resolve_competition_path()
print(f"Resolved DATA_DIR: {Config.DATA_DIR}")
print(f"Target Output Dir: {Config.OUTPUT_DIR}")
print(f"Target Device    : {Config.DEVICE}")

> *Mühendislik Açıklaması*<br>
    1. **`TARGET_COLS` Sıralaması:** Kaggle değerlendirme motoru, tahmin sütunlarının `sample_submission.csv` ile birebir aynı sırada olmasını bekler. Bu liste baştan itibaren sabitlenmiştir.<br>
    2. **`GRAD_ACCUM_STEPS = 2`:** GPU belleği (VRAM) 48 adet 3 kanallı yüksek çözünürlüklü pencereyi partilerken taşmasın diye fiziksel parti boyutu 2 tutulur; 2 adımda bir gradyan güncellenerek bellek taşması riski sıfırlanır.<br>
    3. **`resolve_competition_path()`:** Dizin adı ister `rsna-knee-abnormality-detection` olsun ister farklı bir bağlama adı, train.csv dosyasını arayarak yarışma kökünü otomatik bağlar.

## Bölüm 2: Dinamik Veri Girişi ve 4.407 Vakalık Kohort Yönetimi
Bu ana bölüm, yarışmadaki zayıf denetimli öğrenme verisini sisteme dahil eder. 2.1 alt başlığında dosya yollarının dinamik keşfi yapılır; 2.2 alt başlığında ise etiketler birleştirilerek altın standart ve pseudo-etiketli vakalar birbirinden ayrılır.

### 2.1 Çok Kaynaklı Dizin Taraması (`load_training_cohort`)
Bu hücre, `/kaggle/input` altındaki tüm bağlı veri setlerini özyinelemeli (recursive) olarak tarayarak `train_annotated.csv` dosyasını otomatik olarak keşfeder. 
Statik yol tanımlarından kaynaklanabilecek `FileNotFoundError` hatalarını tamamen önler.

In [ ]:
def locate_annotated_file(data_dir: Path) -> Optional[Path]:
    """
    Kaggle ve yerel çalışma dizinlerindeki olası tüm dosya yollarını tarayarak
    en güncel açıklama (annotation) dosyasını bulur.
    """
    candidates = []
    
    # 1. Kaggle ortamındaki tüm alt dizinlerin özyinelemeli taranması
    if Path('/kaggle/input').exists():
        candidates.extend(list(Path('/kaggle/input').rglob('*train_annotated*.csv')))
        
    # 2. Yerel proje dizini alternatifleri
    candidates.extend([
        data_dir / 'train_annotated.csv',
        Path('data/train_annotated.csv'),
        Path('../data/train_annotated.csv')
    ])
    
    for path in candidates:
        if path.exists():
            return path.resolve()
            
    return None

annotated_file_path = locate_annotated_file(Config.DATA_DIR)
print(f"Keşfedilen Açıklama Dosyası Yolu: {annotated_file_path}")

> *Mühendislik Açıklaması* <br>
    Bu fonksiyon, Kaggle üzerinde yüklenen veri setlerinin oluşturduğu klasörleme hiyerarşisinden bağımsız çalışır. Kullanıcı dosyayı hangi veri seti adı altında yüklemiş olursa olsun (`rsna-knee-annotated`, `my-labels`, vb.), `rglob('*train_annotated*.csv')` çağrısı dosyayı doğrudan yakalar.

### 2.2 Zayıf Denetimli Pseudo-Label Entegrasyonu ve Altın Standart İzolasyonu
Bu hücre, 4.407 hastanın verisini belleğe yükler. Manuel uzman etiketine sahip 58 altın vaka ile LLM tarafından damıtılmış 4.349 zayıf denetimli vakayı birbirinden ayıran `is_gold` mantıksal bayrağını oluşturur ve kohortun sınıf istatistiklerini doğrular.

In [ ]:
def load_and_isolate_cohort(data_dir: Path, target_file: Optional[Path]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Kohortu yükler, 58 altın vaka ile 4.349 pseudo-etiketli vakayı işaretler
    ve seri meta verilerini bağlar.
    """
    train_csv = data_dir / 'train.csv'
    series_csv = data_dir / 'train_series.csv'
    
    if not train_csv.exists() or not series_csv.exists():
        raise FileNotFoundError(f"Yarışma ana veri dosyaları bulunamadı: {data_dir}")
        
    df_raw = pd.read_csv(train_csv)
    df_series = pd.read_csv(series_csv)
    
    # Altın standart insan uzman etiketlerine sahip 58 StudyInstanceUID listesi
    gold_uids = set(df_raw[df_raw['ACL'].notna()]['StudyInstanceUID'].values)
    
    if target_file is not None and target_file.exists():
        print(f"Açıklamalı kohort yükleniyor: {target_file}")
        df_all = pd.read_csv(target_file)
        df_all['is_gold'] = df_all['StudyInstanceUID'].isin(gold_uids)
    else:
        # Fallback: Yalnızca 58 altın vaka
        print("BİLGİ: Açıklamalı dosya bulunamadı, 58 altın standart vaka izole ediliyor.")
        df_all = df_raw[df_raw['ACL'].notna()].copy().reset_index(drop=True)
        df_all['is_gold'] = True
        
    return df_all, df_series

# Kohortun oluşturulması ve ayrımı
df_all, df_series = load_and_isolate_cohort(Config.DATA_DIR, annotated_file_path)

gold_cases = int(df_all['is_gold'].sum())
pseudo_cases = int((~df_all['is_gold']).sum())

print("=======================================================")
print(f"Toplam Yüklenen Hasta İncelemesi (Studies): {len(df_all)}")
print(f"  - Altın Standart (Radyolog Etiketli)    : {gold_cases}")
print(f"  - Zayıf Denetimli (Pseudo-Etiketli)     : {pseudo_cases}")
print(f"Toplam Bağlanan MR Serisi (Series)        : {len(df_series)}")
print("=======================================================")
print("\nHedef Etiketlerin İlk 3 Satırı:")
display(df_all[['StudyInstanceUID', 'is_gold'] + Config.TARGET_COLS[:4]].head(3))

> *Klinik ve İstatistiki Açıklama*<br>
    1. **İzolasyonun Önemi:** 58 altın vaka, yarışmanın test kümesini hazırlayan bağımsız radyolog kurulu tarafından doğrulanmış mutlak gerçektir. Bu vakaların `is_gold=True` olarak etiketlenmesi, ileride modelin eğitim döngüsünde hem aşırı öğrenmeyi (overfitting) hem de etiket gürültüsünü (label noise) denetlememize olanak tanır.<br>
    2. **Doğrulama Ölçütü:**<br>
            - `len(df_all)` değeri **4407**<br>
            - `gold_cases` değeri **58**<br>
            - `pseudo_cases` değeri **4349** olmalıdır. Bu üçlü doğrulandığında, zayıf denetimli veri hattı tamamen güvencededir.

## Bölüm 3: Fiziksel DICOM İşleme Motoru ve 2.5D Pencereleme
Bu ana bölüm, hastane tarayıcılarından (Siemens, GE, Philips) çıkan ham DICOM dosyalarını piksel seviyesinde okur, fiziksel kalibrasyonlarını yapar, farklı tarayıcı çözünürlüklerini standartlaştırır ve 2.5D üç boyutlu tensör temsillerine dönüştürür.

### 3.1 Eğim/Kesişim Kalibrasyonu ve Percentile Normalizasyonu
Bu hücre, ham DICOM piksel dizisini okur. Tarayıcı donanımının meta verilerinde saklanan fiziksel eğim (`RescaleSlope`) ve kesişim (`RescaleIntercept`) değerlerini uygulayarak sinyal yoğunluğunu kalibre eder. MONOCHROME1 fotometrik yorumlamasını tersine çevirir ve aşırı parlak gürültüleri (artefaktları) 1. ve 99. yüzdelik dilimler (percentile) arasında kırpar.

In [ ]:
def calibrate_and_normalize_dicom(dcm_path: Union[str, Path]) -> np.ndarray:
    """
    DICOM dosyasından piksel verisini okur, fiziksel sinyal kalibrasyonunu
    ve yüzde 1-99 aralığında kontrast normalizasyonunu uygular [0.0, 1.0].
    """
    if pydicom is None:
        return np.zeros((Config.IMAGE_SIZE, Config.IMAGE_SIZE), dtype=np.float32)
        
    dcm = pydicom.dcmread(str(dcm_path))
    arr = dcm.pixel_array.astype(np.float32)
    
    # 1. Donanımsal Eğim (Slope) ve Kesişim (Intercept) Düzeltmesi
    slope = getattr(dcm, 'RescaleSlope', 1.0)
    intercept = getattr(dcm, 'RescaleIntercept', 0.0)
    if slope != 1.0 or intercept != 0.0:
        arr = arr * slope + intercept
        
    # 2. MONOCHROME1 Yorumlamasını Düzeltme (Ters Renk / Kemik-Hava Polaritesi)
    if getattr(dcm, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        arr = np.amax(arr) - arr
        
    # 3. Yüzdelik Dilim (Percentile 1-99) Normalizasyonu: Ekstrem gürültü piksellerini eler
    p_low, p_high = np.percentile(arr, (1, 99))
    if p_high > p_low:
        arr = np.clip(arr, p_low, p_high)
        arr = (arr - p_low) / (p_high - p_low)
    else:
        arr = np.zeros_like(arr)
        
    return arr.astype(np.float32)

> *Medikal Fizik Açıklaması*
>
> 1. **RescaleSlope ve RescaleIntercept:**  
>    Manyetik Rezonans (MR) tarayıcıları, sinyal değerlerini tamsayı (integer) formatında saklarken fiziksel sinyal yoğunluğunu bir katsayı ile çarpar:  
>    $$Y = \text{Slope} \times X + \text{Intercept}$$  
>    Bu dönüşüm yapılmadığında farklı tarayıcılardan alınan görüntüler arasında yapay kontrast farkları oluşur.
>
> 2. **MONOCHROME1 vs MONOCHROME2:**  
>    Tıbbi standartlarda pikseller genellikle `MONOCHROME2` (düşük değer = siyah/hava, yüksek değer = beyaz/kemik/sıvı) formatındadır. Bazı cihazlar `MONOCHROME1` üretir; ters çevrilmezse model kemiği hava, havayı kemik sanır.
>
> 3. **%1-%99 Persentil (Percentile) Kırpma:**  
>    MR çekimlerinde metal implantlar veya yağ dokusu piksel değerlerini 4000+ seviyesine fırlatabilir. Klasik min-max normalizasyonu bu durumda tüm kıkırdak dokusunu sıfıra ezer. %1-%99 dilimleme bu ekstrem aykırı değerleri (outliers) yok eder ve doku kontrastını korur.

### 3.2 Değişken MR Çözünürlüklerinin Çift Çizgisel Standartlaştırılması
Aynı hastanın farklı serileri (örneğin Sagittal 960x960, Coronal 800x800) değişken matris boyutlarına sahiptir. Bu hücre, GPU/CPU üzerinde çift çizgisel (bilinear) enterpolasyon uygulayarak tüm dilimleri homojen `(384, 384)` matrisine dönüştürür.

In [ ]:
def load_dicom_slice(dcm_path: Union[str, Path], target_size: int = Config.IMAGE_SIZE) -> np.ndarray:
    """
    DICOM dilimini okur, fiziksel olarak normalize eder ve uzamsal boyutunu
    kesin olarak (target_size, target_size) matrisine enterpole eder.
    """
    arr = calibrate_and_normalize_dicom(dcm_path)
    
    # Uzamsal boyut denetimi ve çift çizgisel (bilinear) boyutlandırma
    if arr.shape != (target_size, target_size):
        tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
        tensor = F.interpolate(
            tensor, 
            size=(target_size, target_size), 
            mode='bilinear', 
            align_corners=False
        )
        arr = tensor.squeeze().numpy()
        
    return arr.astype(np.float32)

> *Mühendislik Açıklaması*<br>
Bu enterpolasyon katmanı, önceki aşamada aldığımız `ValueError: size 960 vs size 800` hatasını kökten çözer. PyTorch konvolüsyonel ağları tensörlerin uzamsal boyutlarının partileme (batching) esnasında tamamen eşit olmasını şart koşar. Bilinear Enterpolasyon, anatomik doku kenarlarını bozmadan piksel yoğunluğunu **384×384** ölçeğine pürüzsüzce indirger.

### 3.3 Üç Düzlemli 2.5D RGB Dilim Yığınlama
MR serisindeki ardışık 3 dilimi [Dilim_{i-1}, Dilim_i, Dilim_{i+1}] RGB kanalları gibi üst üste yığarak 2.5D pencereler oluşturur. Bu temsil, 3D hacimsel bağlamı 2D evrişimli omurgaların (ResNet, CoAtNet) hız ve bellek verimliliğiyle birleştirir.

In [ ]:
# Anatomik Düzlem Kodlaması: Sagittal (0), Coronal (1), Axial (2)
PLANE_MAP = {'Sagittal': 0, 'Coronal': 1, 'Axial': 2}

def create_25d_windows(slices: List[np.ndarray], target_size: int = Config.IMAGE_SIZE) -> np.ndarray:
    """
    Ardışık 2D dilimleri 3 kanallı 2.5D RGB pencereleri haline getirir.
    Çıktı Boyutu: (num_windows, 3, target_size, target_size)
    """
    if len(slices) < 3:
        base = slices[0] if len(slices) > 0 else np.zeros((target_size, target_size), dtype=np.float32)
        triplet = np.stack([base, base, base], axis=0)  # (3, H, W)
        return triplet[np.newaxis, ...]  # (1, 3, H, W)
    
    windows = []
    # Merkez dilim etrafında önceki ve sonraki dilim yığınlanır
    for i in range(1, len(slices) - 1):
        triplet = np.stack([slices[i - 1], slices[i], slices[i + 1]], axis=0)  # (3, H, W)
        windows.append(triplet)
        
    return np.array(windows, dtype=np.float32)

> *Ortopedik Radyoloji Açıklaması*
>
> * **Neden Tek Dilim (2D) Yetersizdir?**  
>   Bir ön çapraz bağ (ACL) veya menisküs yırtığı tek bir MR karesinde sadece siyah bir çizgi olarak görünebilir. Yırtığın anatomik bütünlüğünü anlayabilmek için radyologlar ardışık kesitleri ileri-geri kaydırarak inceler.
>
> * **Neden Tam 3D Hacim Kullanılmıyor?**  
>   3D konvolüsyonel ağlar (3D ResNet, UNet) GPU belleğini saniyeler içinde tüketir ve ImageNet ön eğitimli (pretrained) filtrelerden mahrumdur.
>
> * **2.5D Formülü:**  
>   Üç ardışık dilimi ($[i-1, i, i+1]$) renkli bir fotoğrafın Kırmızı, Yeşil ve Mavi (RGB) kanalları gibi modele vermek; hem 3D derinlik bağlamını korur hem de ResNet'in hazır milyonlarca ön eğitimli filtresini doğrudan kullanmamızı sağlar.

### Bölüm 4: Veri Kümesi (Dataset) ve Tabakalı Katlama (Stratification)

Bu ana bölüm, medikal görüntüleme boru hattını PyTorch'un `Dataset` ve `DataLoader` mimarisine bağlar. Hasta düzeyinde (study-level) çok düzlemli verileri organize eder, değişken pencere sayılarını partileyen özel birleştiriciyi kurar ve veri sızıntısını (data leakage) engelleyen 5 katlı tabakalı çapraz doğrulamayı (Stratified 5-Fold) yapılandırır.

### 4.1 Hasta Düzeyinde `KneeStudyDataset` Boru Hattı
RSNA yarışmasında tahminler dilim veya seri düzeyinde değil, hasta incelemesi (`StudyInstanceUID`) düzeyinde talep edilir. Bu sınıf, bir hastaya ait Sagittal, Coronal ve Axial serileri bir araya getirir; uçlardaki boş hava dilimlerini (%10-%90) budar ve 2.5D pencerelerden oluşan hacim tensörleri üretir.

In [ ]:
class KneeStudyDataset(Dataset):
    """
    Çok düzlemli MR serilerini hasta düzeyinde birleştiren PyTorch Veri Kümesi.
    """
    def __init__(self, df: pd.DataFrame, series_df: pd.DataFrame, data_dir: Path, is_test: bool = False):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.data_dir = data_dir
        self.is_test = is_test
        # Hızlı sorgulama için serileri StudyInstanceUID bazında indeksler
        self.series_by_study = {uid: grp for uid, grp in series_df.groupby('StudyInstanceUID')}
        
    def __len__(self) -> int:
        return len(self.df)
        
    def __getitem__(self, idx: int) -> Dict[str, Union[torch.Tensor, str]]:
        study_id = str(self.df.iloc[idx]['StudyInstanceUID'])
        study_series = self.series_by_study.get(study_id, None)
        
        all_windows = []
        plane_codes = []
        
        if study_series is not None:
            for _, srow in study_series.iterrows():
                series_id = str(srow['SeriesInstanceUID'])
                plane = srow.get('Anatomical_Plane', 'Sagittal')
                plane_code = PLANE_MAP.get(plane, 0)
                
                # Kaggle hiyerarşisindeki olası seri klasör kombinasyonları
                candidates = [
                    self.data_dir / 'train_series' / study_id / series_id,
                    self.data_dir / 'train_series' / series_id,
                    self.data_dir / 'test_series' / study_id / series_id,
                    self.data_dir / study_id / series_id
                ]
                series_folder = next((c for c in candidates if c.exists()), None)
                
                slices = []
                if series_folder is not None:
                    dcm_files = sorted(list(series_folder.glob('*.dcm')))
                    n_files = len(dcm_files)
                    
                    # Periferik Hava Dilimi Budaması (%10 baştan, %10 sondan kırpılır)
                    s_idx = int(n_files * 0.10)
                    e_idx = max(s_idx + 3, int(n_files * 0.90))
                    
                    for fpath in dcm_files[s_idx:e_idx]:
                        try:
                            slices.append(load_dicom_slice(fpath))
                        except Exception:
                            continue
                            
                # 3 ve üzeri geçerli dilim varsa 2.5D pencereler üretilir
                if len(slices) >= 3:
                    w = create_25d_windows(slices)
                    all_windows.append(w)
                    plane_codes.extend([plane_code] * len(w))
                    
        # Herhangi bir nedenle dilim bulunamazsa boru hattını çökertmeyen sentetik koruma tensörü
        if len(all_windows) == 0:
            placeholder = torch.zeros((16, 3, Config.IMAGE_SIZE, Config.IMAGE_SIZE), dtype=torch.float32)
            planes = torch.tensor([0, 1, 2] * 6, dtype=torch.long)[:16]
            final_windows = placeholder
            final_planes = planes
        else:
            concatenated = np.concatenate(all_windows, axis=0)
            # Pencere sayısı belirlenen bütçeyi (48) aşarsa homojen alt-örnekleme yapılır
            if len(concatenated) > Config.MAX_WINDOWS_PER_STUDY:
                sub_indices = np.linspace(0, len(concatenated) - 1, Config.MAX_WINDOWS_PER_STUDY, dtype=int)
                concatenated = concatenated[sub_indices]
                plane_codes = [plane_codes[i] for i in sub_indices]
                
            final_windows = torch.tensor(concatenated, dtype=torch.float32)
            final_planes = torch.tensor(plane_codes, dtype=torch.long)
            
        item = {
            'study_id': study_id,
            'images': final_windows,  # Boyut: (N_windows, 3, 384, 384)
            'planes': final_planes    # Boyut: (N_windows,)
        }
        
        if not self.is_test:
            targets = self.df.iloc[idx][Config.TARGET_COLS].values.astype(np.float32)
            item['targets'] = torch.tensor(targets, dtype=torch.float32)  # Boyut: (12,)
            
        return item

> *Mühendislik Açıklaması*
>
> 1. **Periferik Hava Budaması (%10-%90):**  
>    Bir diz MR çekiminde ilk ve son 3-5 kesit genellikle eklemin tamamen dışındaki boş havayı ve cilt dokusunu gösterir. Bu kesitlerin elenmesi, GPU hesaplama gücünün yalnızca eklem kıkırdağı ve bağlarına yoğunlaşmasını sağlar.
>
> 2. **`MAX_WINDOWS_PER_STUDY = 48`:**  
>    Hastaların toplam dilim sayısı 60 ila 120 arasında değişebilir. `np.linspace` ile homojen 48 pencereye seyreltmek, anatomik eksen boyunca hiçbir kritik kesiti kaçırmadan sabit bir GPU bellek profili sunar.

### 4.2 Özel Parti Birleştirici (`collate_studies`)
Farklı hastalar değişken sayıda pencereye ($N_1 \neq N_2$) sahiptir. Standart PyTorch birleştiricisi boyut uyuşmazlığı nedeniyle çöker. Bu özel fonksiyon, görüntü tensörlerini bellek israfı yapmadan liste halinde birleştirir ve çoklu örneklemli (MIL) partileme sağlar.

In [ ]:
def collate_studies(batch: List[Dict]) -> Dict[str, Union[List[torch.Tensor], torch.Tensor, List[str]]]:
    """
    Değişken pencere boyutlarına sahip hasta çalışmalarını gruplar.
    Görüntü ve düzlem tensörleri liste olarak tutulur; hedefler (targets) yığınlanır.
    """
    study_ids = [b['study_id'] for b in batch]
    images = [b['images'] for b in batch]      # [ (N1, 3, 384, 384), (N2, 3, 384, 384), ... ]
    planes = [b['planes'] for b in batch]      # [ (N1,), (N2,), ... ]
    
    result = {
        'study_ids': study_ids,
        'images': images,
        'planes': planes
    }
    
    if 'targets' in batch[0]:
        # Hedef etiketler standart tensör olarak yığınlanır: (Batch_Size, 12)
        result['targets'] = torch.stack([b['targets'] for b in batch], dim=0)
        
    return result

> *Mühendislik Açıklaması*
>
> Geleneksel derin öğrenmede tensörler sıfırlarla doldurulur (*zero-padding*). Ancak tıbbi görüntülemede boş siyah pikseller eklemek, dikkat (*attention*) mekanizmasını bu yapay piksellere çekerek modelin öğrenme sürecini bozar. Tensörleri liste formatında tutup modele dinamik olarak beslemek; sıfır dolgu kirliliğini önler ve GPU belleğinde gereksiz kayıpların önüne geçer.

### 4.3 5 Katlı Tabakalı Bölümleme ve DataLoader Kurulumu
Bu hücre, 4.407 hastayı `StudyInstanceUID` seviyesinde gruplayarak 5 eşit katlamaya böler. 58 altın standart insan uzman vakasının ve 4.349 zayıf etiketli vakanın oranı tüm katlamalara eşit paylaştırılır. Aktif katlama (Fold 0) için veri yükleyiciler oluşturulur.

In [ ]:
# Rastgelelik tohumu sabitlenir
np.random.seed(42)
df_all['fold'] = -1
gold_mask = df_all['is_gold']

# 1. Aşama: 58 altın vaka 5 katlamaya eşit dağıtılır (~11-12 vaka/fold)
gold_idx = df_all[gold_mask].index.to_numpy().copy()
np.random.shuffle(gold_idx)
for i, idx in enumerate(gold_idx):
    df_all.loc[idx, 'fold'] = i % Config.NUM_FOLDS
    
# 2. Aşama: 4.349 pseudo-etiketli vaka 5 katlamaya eşit dağıtılır (~870 vaka/fold)
pseudo_idx = df_all[~gold_mask].index.to_numpy().copy()
if len(pseudo_idx) > 0:
    np.random.shuffle(pseudo_idx)
    for i, idx in enumerate(pseudo_idx):
        df_all.loc[idx, 'fold'] = i % Config.NUM_FOLDS

# Aktif Katlamanın (Fold 0) Ayrılması
train_df = df_all[df_all['fold'] != Config.ACTIVE_FOLD].reset_index(drop=True)
val_df = df_all[df_all['fold'] == Config.ACTIVE_FOLD].reset_index(drop=True)

# Eğitim ve Doğrulama DataLoader Nesneleri
train_loader = DataLoader(
    KneeStudyDataset(train_df, df_series, Config.DATA_DIR),
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_studies,
    num_workers=Config.NUM_WORKERS
)

val_loader = DataLoader(
    KneeStudyDataset(val_df, df_series, Config.DATA_DIR),
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_studies,
    num_workers=Config.NUM_WORKERS
)

print("=======================================================")
print(f"Aktif Çapraz Doğrulama Katlaması (Fold): {Config.ACTIVE_FOLD} / {Config.NUM_FOLDS}")
print(f"Eğitim Kümesi (Train Studies)          : {len(train_df)}")
print(f"  - Altın Standart Vaka                : {train_df['is_gold'].sum()}")
print(f"Doğrulama Kümesi (Validation Studies)  : {len(val_df)}")
print(f"  - Altın Standart Vaka                : {val_df['is_gold'].sum()}")
print(f"Eğitim Parti Sayısı (Batches per Epoch): {len(train_loader)}")
print("=======================================================")

> *İstatistiki ve Klinik Açıklama*
>
> 1. **Veri Sızıntısı (*Data Leakage*) Koruması:**  
>    Bölümleme seriler üzerinden değil, kesinlikle hasta kimliği (`StudyInstanceUID`) üzerinden yapılır. Bir hastanın Sagital serisi eğitimdeyken Koronal serisi doğrulamaya sızamaz; böylece modelin hastayı ezberlemesi (*overfitting*) engellenir.
>
> 2. **Dengeli Altın Temsili:**  
>    Doğrulama kümesinde (*Validation*) her zaman 882 vaka ve bu vakaların içinde yaklaşık 12 altın standart uzman vaka yer alır. Böylece modelin başarısı hem geniş kohortta hem de uzman onaylı alt grupta güvenle test edilir.

## Bölüm 5: Çok Görünümlü Derin Öğrenme Mimarisi: 12 Başlı Dikkat Havuzlama

Bu ana bölüm, geliştirdiğimiz yapay zeka modelinin mimarisini ve nöral katmanlarını kurar. Evrişimli omurga (ResNet-34) ile anatomik düzlem gömme vektörlerini birleştirir ve her patoloji için MR kesitlerine bağımsız dikkat (attention) atayan 12 başlı dikkat havuzlama mekanizmasını inşa eder.

### 5.1 Evrişimli Omurga ve Düzlem Koşullama (Plane Embedding)
Bu hücre, 2.5D pencerelerden 512 boyutlu görsel öznitelik vektörleri çıkaran ResNet-34 omurgasını ve pencerelerin hangi anatomik açıdan (Sagittal=0, Coronal=1, Axial=2) çekildiğini modele öğreten düzlem gömme katmanını (`nn.Embedding`) yapılandırır.

In [ ]:
def build_feature_extractor(backbone_name: str = Config.BACKBONE, pretrained: bool = Config.PRETRAINED) -> Tuple[nn.Module, int]:
    """
    ResNet-34 evrişimli omurgasını hazırlar ve sınıflandırma katmanını kaldırarak
    öznitelik çıkarıcı haline getirir.
    """
    if timm is not None:
        try:
            backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        except Exception as e:
            print(f"Çevrimdışı moda geçildi ({e}). Rastgele ağırlıklarla başlatılıyor.")
            backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0)
        in_features = backbone.num_features
    else:
        # PyTorch yerleşik TorchVision alternatifi
        weights = tv_models.ResNet34_Weights.DEFAULT if pretrained else None
        res = tv_models.resnet34(weights=weights)
        in_features = res.fc.in_features
        res.fc = nn.Identity()  # Sınıflandırıcı kaldırılır, doğrudan 512 boyutlu vektör alınır
        backbone = res
        
    return backbone, in_features

> *Derin Öğrenme ve Radyoloji Açıklaması*
>
> 1. **Neden ResNet-34?**  
>    Medikal görüntülemede aşırı derin ağlar (ResNet-152, DenseNet-201) yüksek parametre sayıları nedeniyle aşırı öğrenmeye (*overfitting*) yatkındır. ResNet-34; 21 milyon parametresi ve artık bağlantıları (*residual connections*) ile MR sinyallerini yüksek hız ve güçlü genelleme kabiliyetiyle işleyen optimal derinliktedir.
>
> 2. **Düzlem Koşullamanın (*Plane Embedding*) Mantığı:**  
>    Bir diz MR'ında Sagital kesit ile Aksiyel kesit anatomik olarak bambaşka görünür. Bir dilimin özniteliklerine `nn.Embedding(3, 512)` vektörü eklemek, konvolüsyonel öznitelikleri uzamsal olarak yönlendirir; model *"şu anda Sagital düzleme bakıyorum, çapraz bağ aramalıyım"* bağlamını kazanır.

### 5.2 Patolojiye Özel 12 Başlı Dikkat Havuzlama
Ortalama havuzlama (average pooling), 48 kesit içindeki 1-2 yırtık kesiti sağlıklı dokularla seyreltir. Bu modül, 12 anormalliğin her biri için bağımsız dikkat ağırlıkları ($\alpha_k$) hesaplar; ACL'nin Sagittal'e, MCL'nin Coronal'e, Eklem Efizyonunun Axial'e odaklanmasını sağlar.

In [ ]:
class FindingSpecificAttentionPooling(nn.Module):
    """
    12 anormallik için bağımsız dikkat ağırlıkları hesaplayan Çok Başlı Dikkat Katmanı.
    """
    def __init__(self, in_features: int, num_classes: int = Config.NUM_CLASSES, hidden_dim: int = 128):
        super().__init__()
        self.in_features = in_features
        self.num_classes = num_classes
        
        # Dikkat projeksiyon matrisleri: W * tanh(V * f)
        self.attention_v = nn.Linear(in_features, hidden_dim)
        self.attention_w = nn.Linear(hidden_dim, num_classes)
        
        # Her anormallik için bağımsız sınıflandırıcı doğrusal katmanlar
        self.classifiers = nn.ModuleList([
            nn.Linear(in_features, 1) for _ in range(num_classes)
        ])
        
    def forward(self, features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Girdi : features (N_windows, in_features)
        Çıktı : logits (num_classes,), att_weights (num_classes, N_windows)
        """
        # 1. Adım: Dikkat skorları hesaplanır: (N_windows, num_classes)
        att_scores = self.attention_w(torch.tanh(self.attention_v(features)))
        
        # 2. Adım: Pencereler ekseninde Softmax ile normalizasyon: (num_classes, N_windows)
        att_weights = F.softmax(att_scores.transpose(0, 1), dim=-1)
        
        # 3. Adım: Her sınıf için pencerelerin ağırlıklı toplamı (Bağlam Matrisi): (num_classes, in_features)
        context = torch.matmul(att_weights, features)
        
        # 4. Adım: Her sınıf kendi özel bağlam vektöründen tahmin (logit) üretir: (num_classes,)
        logits = torch.stack([
            self.classifiers[k](context[k]).squeeze(-1) for k in range(self.num_classes)
        ], dim=-1)
        
        return logits, att_weights

> *Matematiksel Formülasyon*
>
> Her bir patoloji $k \in \{1, \dots, 12\}$ için $i$. pencerenin dikkat katsayısı ($\alpha_{k, i}$) şu şekilde türetilir:
>
> $$a_{k, i} = \mathbf{w}_k^T \tanh(V f_i)$$
>
> $$\alpha_{k, i} = \frac{\exp(a_{k, i})}{\sum_{j=1}^{N} \exp(a_{k, j})}$$
>
> $$c_k = \sum_{i=1}^{N} \alpha_{k, i} f_i$$
>
> $$\hat{y}_k = \mathbf{w}_k^T c_k + b_k$$
>
> **Bu sayede:**
> * **ACL başlığı** yırtık bağ dilimlerine %95 ağırlık verirken, **kıkırdak aşınması başlığı** eklem yüzeyi dilimlerine %90 ağırlık verebilir.
> * Model, tıpkı bir radyolog gibi önemli dilimleri öne çıkarıp önemsiz dilimlerin etkisini sıfıra indirir.

### 5.3 Bütünleşik Çok Görünümlü Model Sınıfı (`KneeStudyModel`)
Bu hücre, ResNet-34 öznitelik çıkarıcısını, düzlem gömme katmanını ve 12 başlı dikkat havuzlama modülünü tek bir PyTorch `nn.Module` çatısı altında birleştirir.

In [ ]:
class KneeStudyModel(nn.Module):
    """
    Çok Düzlemli ve 12 Başlı Dikkat Havuzlamalı Bütünleşik Hasta Düzeyi Modeli.
    """
    def __init__(
        self, 
        backbone_name: str = Config.BACKBONE, 
        pretrained: bool = Config.PRETRAINED, 
        num_classes: int = Config.NUM_CLASSES
    ):
        super().__init__()
        # 1. Evrişimli Öznitelik Çıkarıcı
        self.backbone, in_features = build_feature_extractor(backbone_name, pretrained)
        
        # 2. Düzlem Gömme Katmanı (Sagittal, Coronal, Axial)
        self.plane_emb = nn.Embedding(3, in_features)
        
        # 3. 12 Başlı Patolojiye Özel Dikkat Havuzlama
        self.pool = FindingSpecificAttentionPooling(in_features=in_features, num_classes=num_classes)
        
    def forward(self, study_images: List[torch.Tensor], study_planes: List[torch.Tensor]) -> Dict[str, torch.Tensor]:
        """
        Partideki tüm hastaların pencerelerini işler ve logit tensörlerini döndürür.
        """
        batch_logits = []
        for img, pl in zip(study_images, study_planes):
            # img: (N_windows, 3, 384, 384) -> feat: (N_windows, 512)
            feat = self.backbone(img)
            
            # pl: (N_windows,) -> plane_vector: (N_windows, 512)
            feat = feat + self.plane_emb(pl)
            
            # logits: (12,)
            logits, _ = self.pool(feat)
            batch_logits.append(logits)
            
        # Çıktı Boyutu: (Batch_Size, 12)
        return {'logits': torch.stack(batch_logits, dim=0)}

# Modelin ilklendirilmesi ve GPU'ya aktarımı
device = torch.device(Config.DEVICE)
model = KneeStudyModel().to(device)

total_params = sum(p.num_grad_elements if hasattr(p, 'num_grad_elements') else p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=======================================================")
print(f"Model Başarıyla Oluşturuldu: {Config.BACKBONE.upper()}")
print(f"Toplam Parametre Sayısı   : {total_params:,}")
print(f"Eğitilebilir Parametreler  : {trainable_params:,}")
print(f"Hedef Cihaz (Target Device): {device}")
print("=======================================================")

> *Mimari Doğrulama Ölçütü*
>
> Bu hücre çalıştırıldığında çıktıda yaklaşık **21.4 milyon parametre** ve `cuda` hedef cihazı görülmelidir. Model GPU belleğine başarıyla yüklendiğinde, bir sonraki aşamada eğitim optimizasyonu devreye alınır.

## Bölüm 6: Kayıp Fonksiyonu, Optimizasyon ve Eğitim Döngüsü

Bu ana bölüm, modelin gradyan hesaplama ve öğrenme mekanizmasını kurar. Tıbbi anormalliklerin görülme sıklıklarındaki dengesizliği gideren pozitif ağırlıklı ikili çapraz entropi (Finding-Weighted BCE) kayıp fonksiyonunu, ağırlık çürümesini ayrıştıran AdamW optimizasyon algoritmasını ve NVIDIA Tensor çekirdeklerini kullanarak GPU hızını iki katına çıkaran karma duyarlık (AMP FP16) eğitim döngüsünü yapılandırır.

### 6.1 Sınıf Dengesizliklerine Duyarlı Kayıp Fonksiyonu
Diz MR'larında patolojilerin görülme sıklıkları homojen değildir (Eklem Sıvısı %60 iken, MCL Yırtığı %15'tir). Standart ikili çapraz entropi (BCE) kullanıldığında model baskın sınıflara odaklanıp nadir bağ yırtıklarını ihmal eder. Bu hücre, her sınıf için ters frekanslı pozitif sınıf ağırlığı (`pos_weight`) hesaplar.

In [ ]:
def calculate_class_weights(df: pd.DataFrame, device: torch.device) -> torch.Tensor:
    """
    12 hedef patoloji için pozitif sınıf ağırlıklarını hesaplar.
    Aşırı cezalandırmayı önlemek için ağırlıklar [1.0, 4.0] sınırında kırpılır.
    """
    weights = []
    total_samples = len(df)
    
    for col in Config.TARGET_COLS:
        pos_samples = (df[col] > 0.5).sum()
        neg_samples = total_samples - pos_samples
        # Sıklık Oranı: Negatif / Pozitif
        ratio = max(1.0, min(neg_samples / max(1, pos_samples), 4.0))
        weights.append(ratio)
        
    return torch.tensor(weights, dtype=torch.float32, device=device)

# Ağırlıkların hesaplanması ve BCEWithLogitsLoss kriterinin oluşturulması
pos_weights = calculate_class_weights(train_df, device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

print("Hesaplanan Pozitif Sınıf Ağırlıkları (pos_weights):")
for col, w in zip(Config.TARGET_COLS, pos_weights.cpu().numpy()):
    print(f"  - {col:<18}: {w:.2f}")

> *Matematiksel ve Klinik Açıklama*
>
> Standart İkili Çapraz Entropi (*Binary Cross Entropy*) formülünde pozitif ($y_k = 1$) ve negatif ($y_k = 0$) örnekler eşit ağırlığa sahiptir. **Bulgu Ağırlıklı BCE (*Finding-Weighted BCE*)** yönteminde ise pozitif örnekler $w_k$ katsayısıyla ölçeklendirilir:
>
> $$\mathcal{L}_k = - \left[ w_k \cdot y_k \log(\sigma(\hat{y}_k)) + (1 - y_k) \log(1 - \sigma(\hat{y}_k)) \right]$$
>
> $$w_k = \min\left(4.0, \, \max\left(1.0, \, \frac{N_{\text{neg}, k}}{N_{\text{pos}, k}}\right)\right)$$
>
> * **Neden Kırpma (*Clamping* $[1.0, 4.0]$)?**  
>   Çok nadir bir sınıfta oran 10 veya 20'ye kadar çıkarsa, model tek bir yanlış pozitifte (*false positive*) aşırı yüksek gradyan üretir ve eğitim kararsızlaşır (*gradient instability*). Maksimum 4.0 üst sınırı bu gradyan patlamasını engeller.
>
> * **`BCEWithLogitsLoss` Sayısal Kararlılığı:**  
>   Sigmoid fonksiyonu ($\sigma$) ile Çapraz Entropi hesabı tek bir *log-sum-exp* işleminde birleştirilerek bilgisayar mimarisindeki sıfıra yaklaşma ve taşma (*underflow/overflow*) hatalarının önüne geçilir.

### 6.2 Optimizasyon ve Öğrenme Oranı Zamanlayıcısı
Model ağırlıklarını güncellemek için ağırlık çürümesini gradyan momentlerinden bağımsız uygulayan AdamW optimizer kullanılır. Öğrenme oranı, eğitim boyunca kosinüs eğrisiyle azalarak ($1\times 10^{-4} \to 1\times 10^{-6}$) modelin pürüzsüz minimumlara oturmasını sağlar.

In [ ]:
# Ağırlık çürümesi ayrıştırılmış (decoupled) AdamW algoritması
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=Config.LEARNING_RATE, 
    weight_decay=Config.WEIGHT_DECAY
)

# Kosinüs Sönümlemeli (Cosine Annealing) Öğrenme Oranı Takvimi
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=Config.EPOCHS, 
    eta_min=1e-6
)

print(f"Optimizer         : AdamW (lr={Config.LEARNING_RATE}, weight_decay={Config.WEIGHT_DECAY})")
print(f"LR Zamanlayıcı    : CosineAnnealingLR (T_max={Config.EPOCHS}, min_lr=1e-6)")

> *Mühendislik Açıklaması*
>
> 1. **AdamW vs. Klasik Adam:**  
>    Klasik Adam optimizasyonunda L2 regülarizasyonu gradyanların içine karışır ve uyarlanabilir adımların büyüklüğünü bozar. AdamW, ağırlık küçültmeyi (*weight decay*) doğrudan parametre güncellemesine uygulayarak derin konvolüsyonel katmanların genelleme yeteneğini belirgin şekilde artırır.
>
> 2. **Kosinüs Sönümleme (*Cosine Annealing*):**  
>    İlk dönemlerde (*epochs*) büyük adımlarla ($10^{-4}$) hata uzayında geniş bölgeler taranır. Son dönemlere gelindiğinde öğrenme oranı $10^{-6}$'ya kadar düşürülerek en keskin ve kararlı yerel minimum vadisine hassas iniş yapılır.

### 6.3 Karma Duyarlık (AMP FP16) ve Gradyan Biriktirmeli Eğitim Fonksiyonu
Bu fonksiyon, tek bir epochluk eğitim döngüsünü yürütür. NVIDIA Tensor çekirdeklerini 16-bit kayan nokta (FP16) modunda çalıştırarak GPU VRAM tüketimini yarıya indirir; gradyan kırpma (clipping) ve gradyan biriktirme (gradient accumulation) ile eğitimi hızlandırır.

In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0
    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

scaler = GradScaler() if device.type == 'cuda' else None

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: Optional[GradScaler],
    device: torch.device,
    grad_accum_steps: int = Config.GRAD_ACCUM_STEPS,
    log_interval: int = 50
) -> float:
    model.train()
    loss_meter = AverageMeter()
    optimizer.zero_grad()
    t_start = time.time()
    total_steps = len(loader)
    
    for step, batch in enumerate(loader):
        study_images = [img.to(device) for img in batch['images']]
        study_planes = [pl.to(device) for pl in batch['planes']]
        targets = batch['targets'].to(device)
        
        use_amp = scaler is not None and device.type == 'cuda'
        with autocast(enabled=use_amp):
            out = model(study_images, study_planes)
            logits = out['logits']
            loss = criterion(logits, targets) / grad_accum_steps
            
        if use_amp:
            scaler.scale(loss).backward()
            if (step + 1) % grad_accum_steps == 0 or (step + 1) == total_steps:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
        else:
            loss.backward()
            if (step + 1) % grad_accum_steps == 0 or (step + 1) == total_steps:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                optimizer.step()
                optimizer.zero_grad()
                
        loss_meter.update(loss.item() * grad_accum_steps, len(study_images))
        
        # CANLI ILERLEME GOSTERGESI: Her 50 adimda bir ekrana basar
        if (step + 1) % log_interval == 0 or (step + 1) == total_steps:
            elapsed = time.time() - t_start
            pct = ((step + 1) / total_steps) * 100
            print(f"  [Egitim] Adim {step + 1:4d}/{total_steps:4d} (%{pct:5.1f}) | Kayip (Loss): {loss_meter.avg:.4f} | Gecen Sure: {elapsed:.0f}s")
            
    return loss_meter.avg


> *Mühendislik Açıklaması*
>
> 1. **`torch.cuda.amp.autocast` ve `GradScaler`:**  
>    Yarım hassasiyetli (FP16) hesaplamalarda çok küçük gradyan değerleri sıfıra yuvarlanıp kaybolabilir (*underflow*). `GradScaler`, kayıp (*loss*) değerini geriye yayılımdan (*backpropagation*) önce geçici bir ölçek katsayısıyla çarparak korur; optimizer güncellemesinden hemen önce tekrar geriye böler. Bu sayede sayısal hassasiyet kaybı yaşamadan ~2 kat hesaplama hızı kazanılır.
>
> 2. **`clip_grad_norm_(max_norm=2.0)`:**  
>    Bir hastanın 48 pencerelik dikkat tensörleri geriye türevlenirken gradyanlar üst üste binip birikebilir. Gradyan normunun 2.0 ile sınırlandırılması, ağırlıkların ani patlamalarla `NaN` değerine dönüşmesini (*exploding gradients*) kesin olarak engeller.
>
> 3. **`grad_accum_steps = 2`:**  
>    GPU belleğine her adımda fiziksel olarak 2 hasta sığdırılır; ancak gradyanlar 2 adım boyunca biriktirildikten sonra tek bir optimizer güncellemesi yapılır. Böylece donanımsal olarak efektif batch boyutu 4 hasta yapılmış gibi kararlı bir gradyan yönü elde edilir.

## Bölüm 7: Doğrulama Kulesi, ROC-AUC Metrikleri ve Ağırlık Kaydı

Bu final bölümü, modelin genelleme başarısını doğrulama (validation) kümesindeki 882 hasta üzerinde test eden değerlendirme kulesini kurar. RSNA yarışmasının resmi değerlendirme metriği olan çok etiketli Macro ROC-AUC skorunu sınıf bazında hesaplar, 5 epochluk ana eğitim döngüsünü yürütür ve en yüksek başarıya ulaşan epoch'un ağırlıklarını /kaggle/working/best_model.pt dosyası olarak diske kaydeder.

### 7.1 Çok Etiketli Doğrulama Kulesi ve Macro ROC-AUC Hesaplayıcı
Yarışmanın resmi değerlendirme metriği, 12 anormalliğin ayrı ayrı ROC-AUC skorlarının aritmetik ortalaması olan Macro ROC-AUC'tur. Bu fonksiyon, doğrulama kümesindeki tüm hastaların tahmin olasılıklarını toplar ve her patoloji için bağımsız AUC skoru türetir.

In [ ]:
def compute_macro_roc_auc(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, Dict[str, float]]:
    class_aucs = {}
    valid_aucs = []
    for idx, col in enumerate(Config.TARGET_COLS):
        yt = y_true[:, idx]
        yp = y_pred[:, idx]
        if len(np.unique(yt)) > 1:
            try:
                score = float(roc_auc_score(yt, yp))
                class_aucs[col] = score
                valid_aucs.append(score)
            except Exception:
                class_aucs[col] = 0.50
        else:
            class_aucs[col] = 0.50
    macro_auc = float(np.mean(valid_aucs)) if len(valid_aucs) > 0 else 0.50
    return macro_auc, class_aucs

def evaluate_model(
    model: nn.Module, 
    loader: DataLoader, 
    criterion: nn.Module, 
    device: torch.device,
    log_interval: int = 100
) -> Tuple[float, float, Dict[str, float], np.ndarray, np.ndarray]:
    model.eval()
    loss_meter = AverageMeter()
    val_preds = []
    val_targets = []
    total_steps = len(loader)
    
    with torch.no_grad():
        for step, batch in enumerate(loader):
            study_images = [img.to(device) for img in batch['images']]
            study_planes = [pl.to(device) for pl in batch['planes']]
            targets = batch['targets'].to(device)
            
            out = model(study_images, study_planes)
            logits = out['logits']
            probs = torch.sigmoid(logits)
            
            loss = criterion(logits, targets)
            loss_meter.update(loss.item(), len(study_images))
            
            val_preds.append(probs.cpu().numpy())
            val_targets.append(targets.cpu().numpy())
            
            if (step + 1) % log_interval == 0 or (step + 1) == total_steps:
                print(f"  [Dogrulama] Adim {step + 1:3d}/{total_steps:3d} | Dogrulama Kaybi: {loss_meter.avg:.4f}")
                
    y_pred = np.concatenate(val_preds, axis=0)
    y_t = np.concatenate(val_targets, axis=0)
    macro_auc, class_aucs = compute_macro_roc_auc(y_t, y_pred)
    return loss_meter.avg, macro_auc, class_aucs, y_pred, y_t


> *Değerlendirme Metriği ve Klinik Analiz*
>
> 1. **Neden Doğruluk (*Accuracy*) Değil de ROC-AUC?**  
>    Tıbbi veri setlerinde sınıflar dengesizdir. Örneğin MCL yırtığı görülme oranı %15'tir. Her hastaya ezbere "sağlam" diyen saf bir model %85 doğruluk (*accuracy*) alır; ancak hasta bireyleri tespit edemez. ROC-AUC ise modelin hastaları doğru sıralama yeteneğini (*discrimination capability*) ölçer; ezberci bir modele kesin olarak 0.50 (yazı-tura seviyesi) puan verir.
>
> 2. **Macro Eşitliği:**  
>    12 patolojinin her biri (ister sık görülen *Effusion*, ister nadir görülen *Fracture* olsun) genel skorda tam $\frac{1}{12}$ paya sahiptir. Bu nedenle her patolojinin bağımsız olarak takip edilmesi şarttır.

### 7.2 Ana Eğitim Yürütücüsü ve Model Ağırlığı Kaydı
Bu hücre, 5 epoch boyunca eğitim ve doğrulama adımlarını art arda çalıştırır. Her epoch'ta öğrenme oranını günceller, en yüksek Macro AUC değerine ulaşıldığında modelin ağırlıklarını `/kaggle/working/best_model.pt` olarak kaydeder.

In [ ]:
best_macro_auc = 0.0
best_checkpoint_path = Config.OUTPUT_DIR / 'best_model.pt'
training_history = []

print("=======================================================")
print(f"  EĞİTİM BAŞLIYOR: FOLD {Config.ACTIVE_FOLD} / {Config.NUM_FOLDS}")
print(f"  Toplam Epoch Sayısı: {Config.EPOCHS}")
print(f"  Eğitim Kümesi: {len(train_df)} | Doğrulama Kümesi: {len(val_df)}")
print(f"  Çıktı Dosyası: {best_checkpoint_path.resolve()}")
print("=======================================================\n")

total_start_time = time.time()

for epoch in range(1, Config.EPOCHS + 1):
    epoch_start_time = time.time()
    
    # 1. Eğitim Adımı
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=device,
        grad_accum_steps=Config.GRAD_ACCUM_STEPS
    )
    
    # 2. Doğrulama Adımı
    val_loss, val_macro_auc, class_aucs, y_pred, y_true = evaluate_model(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device
    )
    
    # 3. Öğrenme Oranı Güncellemesi
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    epoch_duration = time.time() - epoch_start_time
    
    # 4. En İyi Modelin Belirlenmesi ve Kaydedilmesi
    is_best = val_macro_auc > best_macro_auc
    if is_best:
        best_macro_auc = val_macro_auc
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'best_macro_auc': best_macro_auc,
            'class_aucs': class_aucs,
            'backbone': Config.BACKBONE,
            'target_cols': Config.TARGET_COLS
        }, best_checkpoint_path)
        
    print(
        f"Epoch {epoch:02d}/{Config.EPOCHS:02d} [{epoch_duration:.1f}s | lr={current_lr:.1e}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Macro AUC: {val_macro_auc:.4f} "
        f"{'(EN İYİ MODEL KAYDEDİLDİ)' if is_best else ''}"
    )
    
    training_history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_macro_auc': val_macro_auc,
        'class_aucs': class_aucs
    })

total_duration = time.time() - total_start_time

print("\n=======================================================")
print(f"  EĞİTİM BAŞARIYLA TAMAMLANDI!")
print(f"  Toplam Süre       : {total_duration / 60:.1f} dakika")
print(f"  En İyi Macro AUC   : {best_macro_auc:.4f}")
print(f"  Kaydedilen Ağırlık: {best_checkpoint_path.resolve()}")
print("=======================================================")

# En İyi Epoch'un Sınıf Bazlı Detay Raporu
print("\nEn İyi Modelin Patoloji Bazlı Başarım Tablosu:")
best_epoch_classes = max(training_history, key=lambda x: x['val_macro_auc'])['class_aucs']
for finding, score in best_epoch_classes.items():
    print(f"  - {finding:<18}: {score:.4f} AUC")

> *Mühendislik ve Gönderim Doğrulaması*
>
> 1. **`best_model.pt` İçeriği:**  
>    Kaydedilen dosya yalnızca modelin ham ağırlıklarını değil; ulaştığı en yüksek `best_macro_auc` skorunu, 12 patolojinin ayrı ayrı metriklerini ve hedef sütun adlarını bir metadata sözlüğü halinde saklar.
>
> 2. **2. Not Defteri (`02-RSNA-Knee-Inference-Submission`) ile Bütünleşme:**  
>    Eğitim bittiğinde üretilen bu dosya, Kaggle arayüzünden tek tıkla *"Save as Dataset"* seçeneğiyle kaydedilecek ve doğrudan inference notebook'unun girişine (`/kaggle/input/rsna-knee-trained-weights/best_model.pt`) bağlanacaktır.
>